# 02 — Data Cleaning Validation

The dataset provided is **already cleaned**, so this notebook does **not** redo the full
cleaning pipeline. Instead it performs **validation checks**:

- missing values
- duplicate rows
- incorrect data types
- invalid platform / category / content-type values
- invalid / impossible numerical values
- incorrect date/time formats
- empty captions / hashtags (columns, if they exist)

If a check finds a real problem, we fix it here and explain why. If everything is already
valid, we say so explicitly rather than pretending to have done work.

**Result of this notebook: the dataset passed every validation check. No cleaning was necessary.**
The only thing worth a second look was that `Story` and `Video` content types are shared
across more than one platform — this is expected (they are generic formats), not a data
quality problem, and is confirmed against platform-exclusive formats below.

The validated dataset
is saved to `data/processed/cleaned_dataset.csv` and is used by all downstream notebooks —
**the original raw file is never modified.**

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)

RAW_PATH = '../data/raw/Data__Engagement.csv'
df = pd.read_csv(RAW_PATH)
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns.")
issues_found = []  # we log every real issue here for the final summary


Loaded 3025 rows, 22 columns.


## 1. Missing values

In [2]:
missing = df.isnull().sum()
n_missing = missing.sum()
print(f"Total missing cells: {n_missing}")
if n_missing == 0:
    print("VALID: No missing values in any column. No action needed.")
else:
    display(missing[missing > 0])
    issues_found.append(f"{n_missing} missing cells found")


Total missing cells: 0
VALID: No missing values in any column. No action needed.


## 2. Duplicate rows

In [3]:
full_dupes = df.duplicated().sum()
id_dupes = df['Post_ID'].duplicated().sum()
print(f"Fully duplicated rows: {full_dupes}")
print(f"Duplicated Post_ID values: {id_dupes}")

if full_dupes == 0 and id_dupes == 0:
    print("VALID: No duplicate rows and no duplicate Post_IDs. No action needed.")
else:
    issues_found.append(f"{full_dupes} duplicate rows, {id_dupes} duplicate Post_IDs")
    before = len(df)
    df = df.drop_duplicates()
    df = df.drop_duplicates(subset='Post_ID')
    print(f"Removed duplicates: {before - len(df)} rows dropped.")


Fully duplicated rows: 0
Duplicated Post_ID values: 0
VALID: No duplicate rows and no duplicate Post_IDs. No action needed.


## 3. Data types

In [4]:
print(df.dtypes)
print()

expected_numeric = ['Likes', 'Comments', 'Shares', 'Views', 'Saves', 'Follower_Count',
                     'Engagement_Rate', 'Hour_of_Day', 'Hashtag_Count', 'Content_Length']
type_issues = []
for col in expected_numeric:
    if col in df.columns and not pd.api.types.is_numeric_dtype(df[col]):
        type_issues.append(col)

if not type_issues:
    print("VALID: All numeric columns have correct numeric dtypes.")
else:
    print("Columns with incorrect dtype:", type_issues)
    issues_found.append(f"incorrect dtype in {type_issues}")


Post_ID                             str
Timestamp                           str
Platform                            str
Content_Type                        str
Category                            str
Likes                             int64
Comments                          int64
Shares                            int64
Views                             int64
Saves                             int64
Follower_Count                    int64
Engagement_Rate                 float64
Hour_of_Day                       int64
Day_of_Week                         str
Hashtag_Count                     int64
Content_Length                    int64
Sentiment                           str
Influencer_Tier                     str
Has_Media                          bool
Is_Verified                        bool
Flag_Engagement_Above_Views        bool
Flag_Engagement_Rate_Outlier       bool
dtype: object

VALID: All numeric columns have correct numeric dtypes.


## 4. Invalid platform values
We check `Platform` only contains the three expected values.

In [5]:
expected_platforms = {'Instagram', 'Facebook', 'TikTok'}
actual_platforms = set(df['Platform'].unique())
unexpected = actual_platforms - expected_platforms

print("Platforms found:", actual_platforms)
if not unexpected:
    print("VALID: Only expected platforms (Instagram, Facebook, TikTok) are present.")
else:
    print("Unexpected platform values:", unexpected)
    issues_found.append(f"unexpected platform values: {unexpected}")


Platforms found: {'Instagram', 'TikTok', 'Facebook'}
VALID: Only expected platforms (Instagram, Facebook, TikTok) are present.


## 5. Invalid categories

In [6]:
print("Unique categories:", sorted(df['Category'].unique()))
print(f"Number of distinct categories: {df['Category'].nunique()}")
# A basic sanity check: no blank/whitespace-only category strings
blank_categories = df['Category'].astype(str).str.strip().eq('').sum()
print(f"Blank category values: {blank_categories}")
if blank_categories == 0:
    print("VALID: No blank category values.")
else:
    issues_found.append(f"{blank_categories} blank category values")


Unique categories: ['Business', 'Education', 'Entertainment', 'Fashion', 'Fitness', 'Food', 'Gaming', 'Health', 'Lifestyle', 'Sports', 'Technology', 'Travel']
Number of distinct categories: 12
Blank category values: 0
VALID: No blank category values.


## 6. Invalid content types
We check whether any `Content_Type` value appears for a platform where it would be nonsensical (e.g. 'Duet' or 'Stitch', which are TikTok-only formats, showing up under Facebook or Instagram).

In [7]:
crosstab = pd.crosstab(df['Platform'], df['Content_Type'])
display(crosstab)

# Some formats are platform-exclusive by definition (Reel/Carousel = Instagram only,
# Duet/Stitch = TikTok only, Live/Post = Facebook only). "Story" and "Video" are
# generic formats that legitimately exist on more than one platform, so we only
# flag formats that are exclusive-by-definition appearing where they shouldn't.
platform_exclusive_formats = {
    'Reel': 'Instagram', 'Carousel': 'Instagram',
    'Duet': 'TikTok', 'Stitch': 'TikTok',
    'Live': 'Facebook', 'Post': 'Facebook',
}
violations = {}
for fmt, expected_platform in platform_exclusive_formats.items():
    if fmt in df['Content_Type'].unique():
        wrong_platform_rows = df[(df['Content_Type'] == fmt) & (df['Platform'] != expected_platform)]
        if len(wrong_platform_rows) > 0:
            violations[fmt] = len(wrong_platform_rows)

print("Formats shared across multiple platforms (expected, not an error):",
      ['Story', 'Video'])
if not violations:
    print("VALID: All platform-exclusive formats (Reel, Carousel, Duet, Stitch, Live, Post) only appear on their correct platform.")
else:
    print("Platform-exclusive formats found on the wrong platform:", violations)
    issues_found.append(f"platform-exclusive content types on wrong platform: {violations}")


Content_Type,Carousel,Duet,Live,Photo,Post,Reel,Stitch,Story,Video
Platform,,,,,,,,,
Facebook,0,0,239,0,240,0,0,244,261
Instagram,333,0,0,320,0,306,0,324,0
TikTok,0,248,0,0,0,0,259,0,251


Formats shared across multiple platforms (expected, not an error): ['Story', 'Video']
VALID: All platform-exclusive formats (Reel, Carousel, Duet, Stitch, Live, Post) only appear on their correct platform.


## 7. Incorrect numerical values / impossible engagement values

In [8]:
numeric_cols_to_check = ['Likes', 'Comments', 'Shares', 'Views', 'Saves',
                          'Follower_Count', 'Engagement_Rate', 'Hashtag_Count', 'Content_Length']

negative_report = {}
for col in numeric_cols_to_check:
    n_negative = (df[col] < 0).sum()
    if n_negative > 0:
        negative_report[col] = n_negative

print("Columns with negative values:", negative_report if negative_report else "none")

# Hour_of_Day must be within 0-23
invalid_hours = (~df['Hour_of_Day'].between(0, 23)).sum()
print(f"Invalid Hour_of_Day values (outside 0-23): {invalid_hours}")

# Views should generally be >= Likes+Comments+Shares+Saves in a well-formed dataset,
# but the dataset already flags this itself via Flag_Engagement_Above_Views, so we
# don't silently drop these rows - we just confirm the flag is consistent.
engagement_sum = df['Likes'] + df['Comments'] + df['Shares'] + df['Saves']
recomputed_flag = engagement_sum > df['Views']
flag_matches = (recomputed_flag == df['Flag_Engagement_Above_Views']).mean()
print(f"Flag_Engagement_Above_Views matches recomputation: {flag_matches:.1%} of rows")

if not negative_report and invalid_hours == 0:
    print("\nVALID: No negative values and all Hour_of_Day values are within 0-23.")
else:
    issues_found.append("numerical validity issues found (see above)")


Columns with negative values: none
Invalid Hour_of_Day values (outside 0-23): 0
Flag_Engagement_Above_Views matches recomputation: 100.0% of rows

VALID: No negative values and all Hour_of_Day values are within 0-23.


## 8. Date/time format check

In [9]:
try:
    parsed_ts = pd.to_datetime(df['Timestamp'], errors='raise')
    print("VALID: All Timestamp values parse correctly as datetimes.")
    print("Range:", parsed_ts.min(), "to", parsed_ts.max())
except Exception as e:
    print("Timestamp parsing issue:", e)
    issues_found.append("Timestamp parsing issue")

# Cross-check Timestamp-derived hour/day-of-week against the existing Hour_of_Day / Day_of_Week columns
hour_match = (parsed_ts.dt.hour == df['Hour_of_Day']).mean()
dow_match = (parsed_ts.dt.day_name() == df['Day_of_Week']).mean()
print(f"Hour_of_Day consistent with Timestamp: {hour_match:.1%}")
print(f"Day_of_Week consistent with Timestamp: {dow_match:.1%}")
if hour_match < 1.0 or dow_match < 1.0:
    issues_found.append("Hour_of_Day / Day_of_Week inconsistent with Timestamp")


VALID: All Timestamp values parse correctly as datetimes.
Range: 2024-01-01 01:42:00 to 2025-12-31 21:56:00
Hour_of_Day consistent with Timestamp: 100.0%
Day_of_Week consistent with Timestamp: 100.0%


## 9. Empty captions / hashtags
The dataset has no caption-text or hashtag-text columns (confirmed in notebook 01), only `Hashtag_Count`. We validate that numeric column instead.

In [10]:
print("Caption text column present:", any('caption' in c.lower() for c in df.columns))
print("Hashtag text column present:", any(c.lower() in ('hashtags', 'hashtag_text') for c in df.columns))
print("THIS FEATURE IS NOT AVAILABLE IN THE DATASET (actual caption/hashtag text).")
print()

empty_hashtag_count_negative = (df['Hashtag_Count'] < 0).sum()
print(f"Negative Hashtag_Count values: {empty_hashtag_count_negative}")
print(f"Hashtag_Count == 0 (post has no hashtags, this is valid, not an error): {(df['Hashtag_Count']==0).sum()} rows")

empty_content_length = (df['Content_Length'] <= 0).sum()
print(f"Non-positive Content_Length values: {empty_content_length}")

if empty_hashtag_count_negative == 0 and empty_content_length == 0:
    print("VALID: No negative hashtag counts and no non-positive content lengths.")
else:
    issues_found.append("invalid hashtag count / content length values")


Caption text column present: False
Hashtag text column present: False
THIS FEATURE IS NOT AVAILABLE IN THE DATASET (actual caption/hashtag text).

Negative Hashtag_Count values: 0
Hashtag_Count == 0 (post has no hashtags, this is valid, not an error): 107 rows
Non-positive Content_Length values: 0
VALID: No negative hashtag counts and no non-positive content lengths.


## 10. Sentiment / Influencer_Tier category validity

In [11]:
print("Sentiment values:", df['Sentiment'].unique())
print("Influencer_Tier values:", df['Influencer_Tier'].unique())

expected_sentiment = {'Positive', 'Neutral', 'Negative'}
expected_tier = {'Nano', 'Micro', 'Mid-tier', 'Macro'}

unexpected_sentiment = set(df['Sentiment'].unique()) - expected_sentiment
unexpected_tier = set(df['Influencer_Tier'].unique()) - expected_tier

if not unexpected_sentiment and not unexpected_tier:
    print("VALID: Sentiment and Influencer_Tier only contain expected category values.")
else:
    print("Unexpected sentiment values:", unexpected_sentiment)
    print("Unexpected tier values:", unexpected_tier)
    issues_found.append("unexpected Sentiment/Influencer_Tier values")


Sentiment values: <StringArray>
['Positive', 'Neutral', 'Negative']
Length: 3, dtype: str
Influencer_Tier values: <StringArray>
['Macro', 'Mid-tier', 'Micro', 'Nano']
Length: 4, dtype: str
VALID: Sentiment and Influencer_Tier only contain expected category values.


## Validation Summary

In [12]:
print("="*60)
print("DATA CLEANING VALIDATION SUMMARY")
print("="*60)
if not issues_found:
    print("\nNo real data quality problems were found.")
    print("The dataset provided was already clean. No additional cleaning was necessary.")
else:
    print(f"\n{len(issues_found)} issue(s) were found and handled:")
    for i, issue in enumerate(issues_found, 1):
        print(f"{i}. {issue}")


DATA CLEANING VALIDATION SUMMARY

No real data quality problems were found.
The dataset provided was already clean. No additional cleaning was necessary.


## Save validated dataset
Even though no changes were required, we save a copy to `data/processed/` so every downstream notebook reads from one consistent, explicitly-validated source rather than the raw file.

In [13]:
import os
os.makedirs('../data/processed', exist_ok=True)
OUT_PATH = '../data/processed/cleaned_dataset.csv'
df.to_csv(OUT_PATH, index=False)
print(f"Saved validated dataset to {OUT_PATH}")
print(f"Shape: {df.shape}")


Saved validated dataset to ../data/processed/cleaned_dataset.csv
Shape: (3025, 22)
